In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "sans-serif",
    "axes.grid": True,
    "grid.alpha": 0.3,
})
sns.set_palette("tab10")
COLORS = ["#4C72B0","#DD8452","#55A868","#C44E52","#8172B3"]

print("Libraries loaded successfully.")


In [ ]:
CSV_PATH = "data/sample_superstore.csv"


df_raw = pd.read_csv(CSV_PATH, encoding="latin1")

df_raw.columns = [c.lower().replace(" ", "_") for c in df_raw.columns]

df_raw["order_date"] = pd.to_datetime(df_raw["order_date"])
df_raw["ship_date"]  = pd.to_datetime(df_raw["ship_date"])

print(f"Loaded {len(df_raw):,} rows  |  {df_raw.shape[1]} columns")
print(f"Date range: {df_raw['order_date'].min().date()} → {df_raw['order_date'].max().date()}")
df_raw.head(3)


In [ ]:
conn = sqlite3.connect(":memory:")

df_raw.to_sql("superstore", conn, if_exists="replace", index=False)

def sql(query, show=True):
    """Run a SQL query and return a DataFrame; optionally print shape."""
    result = pd.read_sql_query(query, conn)
    if show:
        print(f"  → {result.shape[0]} rows  ×  {result.shape[1]} cols")
    return result

print("SQLite DB ready. Table: superstore")


In [ ]:
schema = sql("PRAGMA table_info(superstore);")
schema[["name","type","notnull","dflt_value"]]


In [ ]:
sql("""SELECT
    COUNT(*)                     AS total_rows,
    COUNT(DISTINCT order_id)     AS unique_orders,
    COUNT(DISTINCT customer_id)  AS unique_customers,
    COUNT(DISTINCT product_id)   AS unique_products,
    COUNT(DISTINCT region)       AS regions,
    COUNT(DISTINCT category)     AS categories,
    COUNT(DISTINCT sub_category) AS sub_categories,
    COUNT(DISTINCT state)        AS states
FROM superstore
""")


In [ ]:
sql("SELECT * FROM superstore LIMIT 5;")


In [ ]:
null_check = sql("""SELECT
    SUM(CASE WHEN order_id     IS NULL THEN 1 ELSE 0 END) AS null_order_id,
    SUM(CASE WHEN sales        IS NULL THEN 1 ELSE 0 END) AS null_sales,
    SUM(CASE WHEN profit       IS NULL THEN 1 ELSE 0 END) AS null_profit,
    SUM(CASE WHEN quantity     IS NULL THEN 1 ELSE 0 END) AS null_quantity,
    SUM(CASE WHEN region       IS NULL THEN 1 ELSE 0 END) AS null_region,
    SUM(CASE WHEN category     IS NULL THEN 1 ELSE 0 END) AS null_category,
    SUM(CASE WHEN customer_id  IS NULL THEN 1 ELSE 0 END) AS null_customer_id
FROM superstore
""")
print("Null values per column:")
null_check


In [ ]:
west = sql("""SELECT order_id, customer_name, city, state,
       ROUND(sales,2) AS sales, ROUND(profit,2) AS profit
FROM   superstore
WHERE  region = 'West'
ORDER  BY sales DESC
LIMIT  15
""")
west


In [ ]:
tech = sql("""SELECT order_id, product_name, sub_category,
       ROUND(sales,2) AS sales, quantity, discount
FROM   superstore
WHERE  category = 'Technology'
ORDER  BY sales DESC
LIMIT  15
""")
tech


In [ ]:
orders_2021 = sql("""SELECT order_id, order_date, customer_name, region,
       ROUND(sales,2) AS sales
FROM   superstore
WHERE  strftime('%Y', order_date) = '2021'
ORDER  BY order_date
LIMIT  15
""")
orders_2021


In [ ]:
high_val = sql("""SELECT order_id, customer_name, product_name,
       ROUND(sales,2) AS sales, ROUND(profit,2) AS profit
FROM   superstore
WHERE  sales > 500
ORDER  BY sales DESC
LIMIT  15
""")
print(f"Orders with sales > $500: {len(high_val)}")
high_val


In [ ]:
heavy_disc = sql("""SELECT order_id, product_name, category,
       ROUND(sales,2) AS sales, discount,
       ROUND(profit,2) AS profit
FROM   superstore
WHERE  discount > 0.3
ORDER  BY discount DESC, profit
LIMIT  15
""")
heavy_disc


In [ ]:
losses = sql("""SELECT order_id, customer_name, product_name,
       ROUND(sales,2) AS sales, discount,
       ROUND(profit,2) AS profit
FROM   superstore
WHERE  profit < 0
ORDER  BY profit
LIMIT  15
""")
total_losses = sql("SELECT COUNT(*) AS n FROM superstore WHERE profit < 0", show=False)
print(f"Total loss-making line items: {total_losses.iloc[0,0]}")
losses


In [ ]:
region_perf = sql("""SELECT
    region,
    COUNT(DISTINCT order_id)                          AS total_orders,
    ROUND(SUM(sales), 2)                              AS total_sales,
    ROUND(SUM(profit), 2)                             AS total_profit,
    ROUND(SUM(profit) / SUM(sales) * 100, 2)         AS profit_margin_pct
FROM superstore
GROUP BY region
ORDER BY total_sales DESC
""")
region_perf


In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
x = range(len(region_perf))
bars = ax.bar([r-0.2 for r in x], region_perf["total_sales"]/1000,
              width=0.35, label="Sales ($K)", color=COLORS[0])
bars2= ax.bar([r+0.2 for r in x], region_perf["total_profit"]/1000,
              width=0.35, label="Profit ($K)", color=COLORS[2])
ax.set_xticks(list(x))
ax.set_xticklabels(region_perf["region"])
ax.set_ylabel("Amount ($K)")
ax.set_title("Regional Sales vs Profit")
ax.legend()
plt.tight_layout()
plt.savefig("outputs/01_region_sales_profit.png", dpi=110)
plt.show()
print("Saved → outputs/01_region_sales_profit.png")


In [ ]:
cat_perf = sql("""SELECT
    category,
    COUNT(*)                                  AS line_items,
    SUM(quantity)                             AS total_qty,
    ROUND(SUM(sales), 2)                      AS total_sales,
    ROUND(SUM(profit), 2)                     AS total_profit,
    ROUND(AVG(discount) * 100, 1)             AS avg_discount_pct
FROM superstore
GROUP BY category
ORDER BY total_sales DESC
""")
cat_perf


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].pie(cat_perf["total_sales"], labels=cat_perf["category"],
            autopct="%1.1f%%", startangle=140, colors=COLORS[:3])
axes[0].set_title("Sales Share by Category")

axes[1].barh(cat_perf["category"], cat_perf["total_profit"]/1000,
             color=COLORS[:3])
axes[1].set_xlabel("Profit ($K)")
axes[1].set_title("Profit by Category")
plt.tight_layout()
plt.savefig("outputs/02_category_breakdown.png", dpi=110)
plt.show()


In [ ]:
subcat = sql("""SELECT
    sub_category,
    category,
    ROUND(SUM(sales), 2)  AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    SUM(quantity)         AS units_sold
FROM superstore
GROUP BY sub_category, category
ORDER BY total_profit DESC
""")
subcat


In [ ]:
seg_perf = sql("""SELECT
    segment,
    COUNT(DISTINCT order_id)   AS orders,
    ROUND(SUM(sales), 2)       AS total_sales,
    ROUND(SUM(profit), 2)      AS total_profit,
    ROUND(AVG(sales), 2)       AS avg_line_sales
FROM superstore
GROUP BY segment
ORDER BY total_sales DESC
""")
seg_perf


In [ ]:
busy_states = sql("""SELECT
    state, region,
    COUNT(DISTINCT order_id) AS order_count,
    ROUND(SUM(sales), 2)     AS total_sales
FROM superstore
GROUP BY state, region
HAVING COUNT(DISTINCT order_id) > 50
ORDER BY order_count DESC
""")
busy_states


In [ ]:
top10_sales = sql("""SELECT
    product_name,
    sub_category,
    ROUND(SUM(sales), 2)  AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    SUM(quantity)         AS units_sold
FROM superstore
GROUP BY product_name, sub_category
ORDER BY total_sales DESC
LIMIT 10
""")
top10_sales


In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
names = [n[:35]+"…" if len(n)>35 else n for n in top10_sales["product_name"]]
bars  = ax.barh(names[::-1], top10_sales["total_sales"][::-1]/1000,
                color=COLORS[0])
ax.bar_label(bars, labels=[f"${v/1000:.1f}K" for v in top10_sales["total_sales"][::-1]],
             padding=4, fontsize=8)
ax.set_xlabel("Total Sales ($K)")
ax.set_title("Top 10 Products by Revenue")
plt.tight_layout()
plt.savefig("outputs/03_top10_products.png", dpi=110)
plt.show()


In [ ]:
top10_profit = sql("""SELECT
    product_name,
    category,
    ROUND(SUM(sales), 2)  AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit
FROM superstore
GROUP BY product_name, category
ORDER BY total_profit DESC
LIMIT 10
""")
top10_profit


In [ ]:
bottom10 = sql("""SELECT
    product_name,
    category,
    ROUND(SUM(sales), 2)  AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit
FROM superstore
GROUP BY product_name, category
ORDER BY total_profit ASC
LIMIT 10
""")
bottom10


In [ ]:
top_cities = sql("""SELECT
    city, state, region,
    ROUND(SUM(sales), 2)     AS total_sales,
    COUNT(DISTINCT order_id) AS orders
FROM superstore
GROUP BY city, state, region
ORDER BY total_sales DESC
LIMIT 10
""")
top_cities


In [ ]:
monthly = sql("""SELECT
    strftime('%Y-%m', order_date)  AS year_month,
    COUNT(DISTINCT order_id)       AS orders,
    ROUND(SUM(sales), 2)           AS monthly_sales,
    ROUND(SUM(profit), 2)          AS monthly_profit
FROM superstore
GROUP BY year_month
ORDER BY year_month
""")
monthly.tail(10)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(monthly["year_month"], monthly["monthly_sales"]/1000,
                alpha=0.25, color=COLORS[0])
ax.plot(monthly["year_month"], monthly["monthly_sales"]/1000,
        color=COLORS[0], marker="o", markersize=3, lw=1.5, label="Sales")
ax.plot(monthly["year_month"], monthly["monthly_profit"]/1000,
        color=COLORS[2], marker="s", markersize=3, lw=1.5, label="Profit")
ax.set_xlabel("Month")
ax.set_ylabel("Amount ($K)")
ax.set_title("Monthly Sales & Profit Trend (2019–2022)")
step = max(1, len(monthly)//12)
ax.set_xticks(range(0, len(monthly), step))
ax.set_xticklabels(monthly["year_month"].iloc[::step], rotation=45, ha="right", fontsize=8)
ax.legend()
plt.tight_layout()
plt.savefig("outputs/04_monthly_trend.png", dpi=110)
plt.show()


In [ ]:
yoy = sql("""SELECT
    strftime('%Y', order_date) AS year,
    category,
    ROUND(SUM(sales), 2)       AS total_sales,
    ROUND(SUM(profit), 2)      AS total_profit
FROM superstore
GROUP BY year, category
ORDER BY year, total_sales DESC
""")
yoy


In [ ]:
top_customers = sql("""SELECT
    customer_id, customer_name, segment, region,
    COUNT(DISTINCT order_id)   AS total_orders,
    ROUND(SUM(sales), 2)       AS lifetime_sales,
    ROUND(SUM(profit), 2)      AS lifetime_profit,
    ROUND(AVG(sales), 2)       AS avg_order_value
FROM superstore
GROUP BY customer_id, customer_name, segment, region
ORDER BY lifetime_sales DESC
LIMIT 10
""")
top_customers


In [ ]:
disc_impact = sql("""SELECT
    CASE
        WHEN discount = 0         THEN '0 — No Discount'
        WHEN discount <= 0.1      THEN '1 — Up to 10%'
        WHEN discount <= 0.2      THEN '2 — 11–20%'
        WHEN discount <= 0.3      THEN '3 — 21–30%'
        WHEN discount <= 0.5      THEN '4 — 31–50%'
        ELSE                           '5 — Above 50%'
    END AS discount_bucket,
    COUNT(*)                              AS line_items,
    ROUND(SUM(profit), 2)                 AS total_profit,
    ROUND(SUM(profit)/SUM(sales)*100, 2)  AS margin_pct
FROM superstore
GROUP BY discount_bucket
ORDER BY discount_bucket
""")
disc_impact


In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
colors = [COLORS[2] if m >= 0 else COLORS[3] for m in disc_impact["margin_pct"]]
bars = ax.bar(disc_impact["discount_bucket"], disc_impact["margin_pct"], color=colors)
ax.axhline(0, color="black", lw=0.8, linestyle="--")
ax.bar_label(bars, labels=[f"{v:.1f}%" for v in disc_impact["margin_pct"]], padding=3, fontsize=8)
ax.set_xlabel("Discount Bucket")
ax.set_ylabel("Profit Margin (%)")
ax.set_title("Profit Margin vs Discount Level")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig("outputs/05_discount_margin.png", dpi=110)
plt.show()


In [ ]:
quarterly = sql("""SELECT
    strftime('%Y', order_date) AS year,
    CASE
        WHEN CAST(strftime('%m', order_date) AS INT) BETWEEN 1 AND 3  THEN 'Q1'
        WHEN CAST(strftime('%m', order_date) AS INT) BETWEEN 4 AND 6  THEN 'Q2'
        WHEN CAST(strftime('%m', order_date) AS INT) BETWEEN 7 AND 9  THEN 'Q3'
        ELSE 'Q4'
    END AS quarter,
    ROUND(SUM(sales), 2)     AS quarterly_sales,
    ROUND(SUM(profit), 2)    AS quarterly_profit,
    COUNT(DISTINCT order_id) AS orders
FROM superstore
GROUP BY year, quarter
ORDER BY year, quarter
""")
quarterly


In [ ]:
pivot = quarterly.pivot(index="year", columns="quarter", values="quarterly_sales")
fig, ax = plt.subplots(figsize=(7,4))
sns.heatmap(pivot/1000, annot=True, fmt=".1f", cmap="YlGnBu",
            linewidths=0.5, ax=ax, cbar_kws={"label":"Sales ($K)"})
ax.set_title("Quarterly Sales Heatmap ($K)")
plt.tight_layout()
plt.savefig("outputs/06_quarterly_heatmap.png", dpi=110)
plt.show()


In [ ]:
ship_speed = sql("""SELECT
    ship_mode,
    COUNT(*)                                                               AS shipments,
    ROUND(AVG(julianday(ship_date) - julianday(order_date)), 2)           AS avg_days,
    MIN(CAST(julianday(ship_date) - julianday(order_date) AS INT))        AS min_days,
    MAX(CAST(julianday(ship_date) - julianday(order_date) AS INT))        AS max_days
FROM superstore
GROUP BY ship_mode
ORDER BY avg_days
""")
ship_speed


In [ ]:
loss_customers = sql("""SELECT
    customer_name, segment,
    COUNT(DISTINCT order_id) AS orders,
    ROUND(SUM(sales), 2)     AS total_sales,
    ROUND(SUM(profit), 2)    AS total_profit
FROM superstore
GROUP BY customer_id, customer_name, segment
HAVING SUM(profit) < 0
ORDER BY total_profit ASC
LIMIT 10
""")
loss_customers


In [ ]:
region_rank = sql("""SELECT
    region,
    ROUND(SUM(sales), 2)                        AS total_sales,
    ROUND(SUM(profit), 2)                       AS total_profit,
    ROUND(SUM(profit)/SUM(sales)*100, 2)        AS margin_pct,
    RANK() OVER (ORDER BY SUM(profit) DESC)     AS profit_rank
FROM superstore
GROUP BY region
ORDER BY profit_rank
""")
region_rank


In [ ]:
num_stats = sql("""SELECT
    'sales'    AS metric, ROUND(MIN(sales),2) AS min_val,
    ROUND(MAX(sales),2) AS max_val, ROUND(AVG(sales),2) AS avg_val,
    ROUND(SUM(sales),2) AS total
FROM superstore
UNION ALL
SELECT 'profit', ROUND(MIN(profit),2), ROUND(MAX(profit),2),
    ROUND(AVG(profit),2), ROUND(SUM(profit),2)
FROM superstore
UNION ALL
SELECT 'quantity', MIN(quantity), MAX(quantity),
    ROUND(AVG(quantity),2), SUM(quantity)
FROM superstore
UNION ALL
SELECT 'discount', MIN(discount), MAX(discount),
    ROUND(AVG(discount),3), NULL
FROM superstore
""")
num_stats


In [ ]:
checks = {
    "Negative sales":       sql("SELECT COUNT(*) AS n FROM superstore WHERE sales < 0", show=False).iloc[0,0],
    "Invalid discount":     sql("SELECT COUNT(*) AS n FROM superstore WHERE discount < 0 OR discount > 1", show=False).iloc[0,0],
    "Ship before order":    sql("SELECT COUNT(*) AS n FROM superstore WHERE julianday(ship_date) < julianday(order_date)", show=False).iloc[0,0],
    "Zero/negative qty":    sql("SELECT COUNT(*) AS n FROM superstore WHERE quantity <= 0", show=False).iloc[0,0],
    "Duplicate row IDs":    sql("SELECT COUNT(*) FROM (SELECT row_id FROM superstore GROUP BY row_id HAVING COUNT(*)>1)", show=False).iloc[0,0],
}
integrity_df = pd.DataFrame(list(checks.items()), columns=["Check","Issues Found"])
integrity_df["Status"] = integrity_df["Issues Found"].apply(lambda x: "✅ PASS" if x==0 else "⚠️ FLAG")
integrity_df


In [ ]:
sql("""SELECT
    strftime('%Y', order_date) AS year,
    COUNT(DISTINCT order_id)   AS orders,
    ROUND(SUM(sales), 2)       AS sales,
    ROUND(SUM(profit), 2)      AS profit
FROM superstore
GROUP BY year
ORDER BY year
""")
